Testing 2 on present arms RL

Verify versions, cuda, etc

In [ ]:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [28]:
import pathlib
NOTEBOOK_DIR = pathlib.Path().resolve()
ROOT = NOTEBOOK_DIR
ROBO_PATH = ROOT / ".." / "external_pkgs" / "RoboSuite"
REQ_PATH_1 = (ROOT / ".." / "external_pkgs" / "RoboSuite" / "requirements.txt").resolve()
REQ_PATH_2 = (ROOT / ".." / "external_pkgs" / "RoboSuite" / "requirements-extra.txt").resolve()

import os, sys
sys.path.insert(0, str(ROBO_PATH))

# print(ROBO_PATH)

In [ ]:
# !{sys.executable} -m pip install -r {REQ_PATH_1} # don't need this anymore if you just install requirements.txt
!{sys.executable} -m pip install -r {REQ_PATH_2}

Obtaining file:///home/arhim/Documents/rover/LearnFlake/src/rl_autonomy (from -r /home/arhim/Documents/rover/LearnFlake/src/external_pkgs/RoboSuite/requirements.txt (line 1))

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: /home/arhim/Documents/rover/LearnFlake/src/rl_autonomy/.rl/bin/python -m pip install --upgrade pip
ERROR: file:///home/arhim/Documents/rover/LearnFlake/src/rl_autonomy (from -r /home/arhim/Documents/rover/LearnFlake/src/external_pkgs/RoboSuite/requirements.txt (line 1)) does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 7.2 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 11.4 MB/s eta 0:00:00 MB/s eta 0:00:01:01

[notice] A new release of p

In [31]:
# Importing robosuite as submodule
import robosuite as suite
from robosuite import make
from robosuite.wrappers import GymWrapper

/home/arhim/Documents/rover/LearnFlake/src/rl_autonomy/../external_pkgs/RoboSuite/robosuite/__init__.py:45: SyntaxWarning: invalid escape sequence '\ '
  __logo__ = """


ModuleNotFoundError: No module named 'cv2'

In [ ]:
import torch
import numpy as np
import gymnasium as gym
from gymnasium import spaces

torch.__version__ , torch.cuda.is_available()

: 

In [ ]:
class RobosuiteGymWrapper(gym.Env):
    """
    Generic wrapper for using robosuite environment with Gymnasium for training reinforcement leanring (RL) policies
    """
    def __init__(self, env_name="Lift", robots="rover2025", has_renderer=False, use_camera_obs=False):
        super().__init__()
        self.env = GymWrapper(
            make(
                env_name=env_name, # Specify task or custom tasks
                robots=robots, # Robot(s) used
                has_renderer=has_renderer, # Toggles on-screen rendering
                has_offscreen_renderer=False if has_renderer else True, # Enables offscreen rendering for faster training (needs to be on if on-screen is off)
                use_camera_obs=use_camera_obs, # Toggles camera-based observations e.g. camera_names=["frontview"] TODO: wrapper with cameras
                reward_shaping=True, # e.g. partial rewards for sub-tasks
                # lite_physics=True
                # renderer="egl", # TODO: make compatible with OpenCV
            ) 
        ) # Converts robosuite env into a gym env

        self.action_space = self.env.action_space # TODO: custom robot action space
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=self.env.observation_space.shape, dtype=np.float32
        ) # TODO: add/preprocess observations ; sensors
    def reset(self, seed=None, options=None):
        """
        Reset the env with optional seeding.

        Args: 
            seed (int, optional): seed for reproducibility.
            options (dict, optional): additional options for reset (NotImplemented).
        
        Returns:
            obs (np.ndarray): initial observation
        """
        if seed is not None:
            self.env.seed(seed) # set seed in robosuite env

        return self.env.reset() # Method to reset env TODO: custom reset logic

    def step(self, action):
        result = self.env.step(action)  # Get the full output
        return result
    
    def render(self, mode="human"):
        return self.env.render() # TODO: custom rendering for specific cameras, offline rl
    
    def close(self):
        self.env.close() # Cleanup

In [4]:
# %pip install stable_baselines3 

In [5]:
# %pip install tqdm

In [3]:
from tqdm import tqdm
import time

# Training loop with progress
def train_with_progress(model, total_timesteps, log_interval=1000):
    steps=0
    with tqdm(total=total_timesteps, desc="Progress", unit="step") as pbar:
        # Train for a chunk fo steps
        while steps < total_timesteps:
            model.learn(total_timesteps=log_interval, reset_num_timesteps=False)
            steps += log_interval
            pbar.update(log_interval)
        # time.sleep(0.1)

Train with DDPG


In [4]:
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
import numpy as np
import os
import datetime

# Init env
env_name = "Lift"
robot_name = "Jaco"
env = RobosuiteGymWrapper(env_name=env_name, robots=robot_name, has_renderer=True)

# Add action noise for exploration
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1*np.ones(n_actions))

# Init model
model = DDPG("MlpPolicy", env, action_noise=action_noise, verbose=1, learning_rate=1e-3)

# Train the model
train_with_progress(model, total_timesteps=10000) # Test trial

# Save model
date = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
save_dir = f"./models/{env_name}/{robot_name}/{date}"
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, "ddpg_pp"))
print(f"Model Saved to: {save_dir}/ddpg_pp.zip")


[robosuite INFO] Loading controller configuration from: C:\Users\camer\robosuite\robosuite\controllers\config\default\composite\basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does 

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Progress:   0%|          | 0/10000 [00:00<?, ?step/s][robosuite INFO] Loading controller configuration from: C:\Users\camer\robosuite\robosuite\controllers\config\default\composite\basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has d

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | 1.61     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 8        |
|    time_elapsed    | 111      |
|    total_timesteps | 4000     |
| train/             |          |
|    actor_loss      | -0.3     |
|    critic_loss     | 0.000283 |
|    learning_rate   | 0.001    |
|    n_updates       | 3899     |
---------------------------------


Progress:  40%|████      | 4000/10000 [06:46<10:30,  9.52step/s][robosuite INFO] Loading controller configuration from: C:\Users\camer\robosuite\robosuite\controllers\config\default\composite\basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The c

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | 1.93     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 7        |
|    time_elapsed    | 133      |
|    total_timesteps | 8000     |
| train/             |          |
|    actor_loss      | -0.74    |
|    critic_loss     | 0.00148  |
|    learning_rate   | 0.001    |
|    n_updates       | 7899     |
---------------------------------


Progress:  80%|████████  | 8000/10000 [14:57<04:06,  8.11step/s][robosuite INFO] Loading controller configuration from: C:\Users\camer\robosuite\robosuite\controllers\config\default\composite\basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The c

Model Saved to: ./models/Lift/Jaco/2025-10-10_17-48-34/ddpg_pp.zip


In [8]:
from robosuite import make

env = make(
    env_name="PickPlace",
    robots="Jaco",
    has_renderer=True,  # Enable visible rendering
    has_offscreen_renderer=False,  # Disable offscreen rendering
    use_camera_obs=False,  # Disable camera observations
    reward_shaping=True,
    lite_physics=True
)
env.reset()
env.render()


[robosuite INFO] Loading controller configuration from: C:\Users\camer\robosuite\robosuite\controllers\config\default\composite\basic.json (composite_controller_factory.py:121)


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

In [9]:
import mujoco
mujoco.__version__

'3.2.6'